# Pydantic model

In [8]:
from pydantic import BaseModel, Field
from typing import Literal
from pydantic_ai import Agent

class RestaurantSearcher(BaseModel): 
    name: str = Field(description="Restaurant name")
    type_of_food: str = Field(description="Type of food or cuisine")
    price_level: Literal["$", "$$", "$$$", "$$$$"] = Field(description="Approximate price range")
    rating: Literal["1", "2", "3", "4", "5"]
    short_description: str = Field(description="Short summary about the type of cuisine the restaurant offers")
    opening_hours: str = Field(description="Opening hours as text")
    location: str = Field(description="Restaurant location/address/area")

class RestaurantList(BaseModel):
    restaurants: list[RestaurantSearcher] = Field(
        min_length=5,
        max_length=5,
        description="Exactly 5 restaurant suggestions"
    )

restaurant_searcher_agent = Agent(
    "openrouter:nvidia/nemotron-3-super-120b-a12b:free",
    system_prompt=""" 
You are a restaurant recommendation assistant. 
The user gives you a location.
Return exactly 5 restaurants near that place. 
It is allowed to invent restaurant if needed, 
but they should still look realistic and relevant to the location.
Vary cuisine types.
Keep descriptions short and useful.
Ratings must be between 1 and 5.
Price level must be one of: $, $$, $$$, $$$$.

""", output_type=RestaurantList
)


In [11]:
location = "Södermalm, Stockholm"

result = await restaurant_searcher_agent.run(location)

result.output

RestaurantList(restaurants=[RestaurantSearcher(name='Fika & Söta', type_of_food='Swedish café / Fika', price_level='$$', rating='4', short_description='Cozy spot for traditional Swedish fika with cinnamon buns and strong coffee.', opening_hours='Mon-Fri 7:30am-7pm, Sat-Sun 8am-8pm', location='Hornsgatan 28, Södermalm, Stockholm'), RestaurantSearcher(name='Söder Sushi', type_of_food='Japanese sushi', price_level='$$$', rating='5', short_description='Fresh nigiri and creative rolls in a minimalist setting.', opening_hours='Tue-Sun 12pm-10pm, Mon closed', location='Folkungagatan 74, Södermalm, Stockholm'), RestaurantSearcher(name='La Taverna', type_of_food='Italian trattoria', price_level='$$', rating='4', short_description='Homemade pasta and wood-fired pizza in a warm, rustic ambience.', opening_hours='Daily 11am-11pm', location='Skånegatan 45, Södermalm, Stockholm'), RestaurantSearcher(name='El Camino', type_of_food='Mexican street food', price_level='$', rating='3', short_description=

In [21]:
def get_restaurants(location: str) -> RestaurantList:
    prompt = f"Suggest 5 restaurants near {location}."
    result = restaurant_searcher_agent.run_sync(prompt)
    return result.output

In [ ]:
restaurants = get_restaurants("Gothenburg city center")
restaurants